In [0]:
%run "/Workspace/Repos/shoyofromconcrete@gmail.com/MultiChannelDataPipeLine/config"



In [0]:
import logging
import time
import traceback
import builtins

from pyspark.sql import functions as F

# Logger
logger = logging.getLogger(CONFIG["logging"]["logger_name"])
logger.setLevel(CONFIG["logging"]["log_level"])

if not logger.handlers:
    handler = logging.StreamHandler()
    formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
    handler.setFormatter(formatter)
    logger.addHandler(handler)

# Set context
spark.sql(f"USE CATALOG {CONFIG['catalog']}")
spark.sql(f"USE SCHEMA {CONFIG['schemas']['gold']}")

In [0]:
def read_silver():

    return spark.table(
        f"{CONFIG['catalog']}.{CONFIG['schemas']['silver']}.sales_silver"
    )

In [0]:
def build_gold_metrics(df):

    return (
        df

        # Null handling first
        .withColumn("discount", F.coalesce(F.col("discount"), F.lit(0)))
        .withColumn("tax", F.coalesce(F.col("tax"), F.lit(0)))
        .withColumn("quantity", F.coalesce(F.col("quantity"), F.lit(0)))

        # Metrics
        .withColumn("final_amount",
                    F.col("total_amount") - F.col("discount") - F.col("tax"))

        .withColumn("gross_amount",
                    F.col("price_per_unit") * F.col("quantity"))

        .withColumn("gross_final_diff",
                    F.col("gross_amount") - F.col("final_amount"))

        # Date features
        .withColumn("order_month", F.month("order_date"))
        .withColumn("order_year", F.year("order_date"))

        # Category
        .withColumn(
            "order_value_category",
            F.when(F.col("final_amount") < 1000, "Low")
             .when(F.col("final_amount").between(1000, 5000), "Medium")
             .otherwise("High")
        )

        # Valid order flag
        .withColumn(
            "is_valid_order",
            F.when(
                (F.col("order_status") != "Cancelled") &
                (F.col("payment_status") != "Failed") &
                (F.col("final_amount") > 0) &
                (F.col("quantity") > 0),
                "Valid"
            ).otherwise("Invalid")
        )
    )

In [0]:
def fill_nulls_by_type(df):

    fill_dict = {}

    for col_name, dtype in df.dtypes:

        if any(t in dtype for t in ["int", "bigint", "double", "float", "decimal"]):
            fill_dict[col_name] = 0

        elif dtype == "string":
            fill_dict[col_name] = "unknown"

    return df.fillna(fill_dict)

In [0]:
def build_fact_sales(df):

    final_cols = [
        "order_id",
        "customer_name",
        "email",
        "phone",
        "product_name",
        "category",
        "quantity",
        "price_per_unit",
        "gross_amount",
        "discount",
        "tax",
        "final_amount",
        "payment_method",
        "payment_status",
        "order_status",
        "order_date",
        "delivery_date",
        "order_month",
        "order_year",
        "shipping_city",
        "pincode",
        "source",
        "order_value_category",
        "is_valid_order",
        "source_file"
    ]

    return df.select(final_cols)

In [0]:
def write_gold(df):

    table_name = f"{CONFIG['catalog']}.{CONFIG['schemas']['gold']}.fact_sales"

    try:
        start_time = time.time()

        logger.info(f"[START] Gold write | table={table_name}")

        (
            df.write
              .format("delta")
              .mode("overwrite")
              .option("mergeSchema", "true")
              .saveAsTable(table_name)
        )

        duration = builtins.round(time.time() - start_time, 2)

        if CONFIG["logging"]["track_execution_time"]:
            logger.info(
                f"[END] Gold write | table={table_name} | {duration} sec"
            )

    except Exception:
        logger.error(f"[ERROR] Gold write failed | table={table_name}")
        logger.error(traceback.format_exc())
        raise

In [0]:
def run_gold():

    try:
        logger.info("[START] Gold Pipeline")

        # Step 1: Read
        df = read_silver()

        # Step 2: Transform
        df = build_gold_metrics(df)

        # Step 3: Clean
        df = fill_nulls_by_type(df)

        # Step 4: Build fact table
        fact_df = build_fact_sales(df)

        # Step 5: Filter good data out
        fact_df = fact_df.filter(F.col("is_valid_order") == "Valid")

        # Step 6: Write
        write_gold(fact_df)

        logger.info("[END] Gold Pipeline SUCCESS")

    except Exception:
        logger.error("[FAILED] Gold Pipeline")
        logger.error(traceback.format_exc())
        raise

In [0]:
run_gold()

In [0]:
%sql
select * from multidatadumps.gold.fact_sales